# FraudShield Colab Training Notebook

This notebook trains an **open-source LLM policy** for FraudShield using a two-stage curriculum:

1. **Public fraud-data adaptation** from a Hugging Face dataset
2. **FraudShield policy adaptation** from environment-compatible action traces

The goal is to learn more than a static heuristic by giving the model broader fraud signals first, then teaching it how to act inside the FraudShield workflow.


In [ ]:
%pip uninstall -y unsloth unsloth_zoo trl transformers tokenizers
%pip install -q openenv-core datasets peft accelerate sentencepiece matplotlib pandas
%pip install -q "transformers==4.51.3" "trl==0.19.1"
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

%cd /content
!rm -rf Fraudshield
!git clone https://github.com/DevikaJ2005/Fraudshield.git
%cd /content/Fraudshield
!ls
%pip install -q -e .


In [ ]:
import os
from getpass import getpass

from huggingface_hub import login

token = getpass('Enter your HF token (optional but recommended): ')
if token.strip():
    os.environ['HF_TOKEN'] = token.strip()
    login(token=token.strip())
    print('HF login completed.')
else:
    print('Skipping HF login for now.')


In [ ]:
import torch

print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available. In Colab, set Runtime > Change runtime type > GPU, then restart.')


In [ ]:
import json
import os
import random
import subprocess
from datetime import datetime

import pandas as pd
from datasets import Dataset, load_dataset

from fraudshield_env import FraudShieldEnvironment
from llm_agent import SnapshotCalibratedFraudDetectionAgent

env = FraudShieldEnvironment(data_path='data', seed=42)
assert env.load_data(), 'FraudShield snapshot failed to load.'
print('FraudShield loaded:', env.data_loader.get_bundle_summary())

random.seed(42)

CANONICAL_ALIASES = [
    'merchant_profile',
    'customer_profile',
    'network_graph',
    'payment_trace',
    'policy_review',
]


def serialize_observation(observation):
    return json.dumps(
        {
            'case_id': observation.case_id,
            'task_name': observation.task_name.value,
            'current_screen': observation.current_screen.value,
            'visible_panels': observation.visible_panels,
            'revealed_evidence': observation.revealed_evidence,
            'linked_case_ids': observation.linked_case_ids,
            'remaining_steps': observation.remaining_steps,
            'remaining_sla': observation.remaining_sla,
            'note_required': observation.note_required,
            'allowed_actions': [action.value for action in observation.allowed_actions],
            'case_summary': observation.case_summary.model_dump(mode='json'),
            'app_context': observation.app_context,
        },
        sort_keys=True,
    )


def prompt_from_observation(observation):
    available = observation.app_context.get('available_investigations', CANONICAL_ALIASES)
    return (
        'You are a fraud analyst operating in a simulated investigation workflow. '
        'Only use visible evidence. Return JSON only.\n\n'
        f'Visible observation:\n{serialize_observation(observation)}\n\n'
        f'Valid investigation aliases: {available}.\n'
        'Respond with JSON using this schema: '
        '{"action_type":"investigate|decide","investigation_target":"alias_or_null","decision":"fraud|legitimate|null","confidence":0.0,"reasoning":"one sentence"}'
    )


def action_to_payload(action):
    action_name = action.action_type.value
    if action_name == 'fetch_merchant_profile':
        return {'action_type': 'investigate', 'investigation_target': 'merchant_profile', 'decision': None, 'confidence': None, 'reasoning': action.reasoning or 'Review seller risk signals before routing.'}
    if action_name == 'fetch_customer_profile':
        return {'action_type': 'investigate', 'investigation_target': 'customer_profile', 'decision': None, 'confidence': None, 'reasoning': action.reasoning or 'Review buyer risk signals before routing.'}
    if action_name == 'fetch_network_graph':
        return {'action_type': 'investigate', 'investigation_target': 'network_graph', 'decision': None, 'confidence': None, 'reasoning': action.reasoning or 'Check linked network risk before routing.'}
    if action_name == 'review_transaction':
        return {'action_type': 'investigate', 'investigation_target': 'payment_trace', 'decision': None, 'confidence': None, 'reasoning': action.reasoning or 'Inspect payment and fulfillment details first.'}
    if action_name == 'check_policy':
        return {'action_type': 'investigate', 'investigation_target': 'policy_review', 'decision': None, 'confidence': None, 'reasoning': action.reasoning or 'Check routing policy before a final decision.'}
    if action_name == 'add_case_note':
        return {'action_type': 'decide', 'investigation_target': None, 'decision': 'fraud', 'confidence': 0.55, 'reasoning': action.note_text or 'Document the case before final routing.'}

    decision = 'fraud' if action.resolution.value in {'block', 'hold', 'escalate'} else 'legitimate'
    confidence = 0.9 if action.resolution.value in {'approve', 'block'} else 0.7
    return {
        'action_type': 'decide',
        'investigation_target': None,
        'decision': decision,
        'confidence': confidence,
        'reasoning': action.reasoning or f'Final routing is {action.resolution.value}.',
    }


def build_fraudshield_rollout_dataset(per_task_episodes=24):
    agent = SnapshotCalibratedFraudDetectionAgent()
    rows = []
    for task_name in ('easy', 'medium', 'hard'):
        for _ in range(per_task_episodes):
            reset_result = env.reset(task=task_name)
            observation = reset_result.observation
            done = False
            while not done:
                action = agent.decide(observation)
                payload = action_to_payload(action)
                rows.append({
                    'text': prompt_from_observation(observation) + '\n' + json.dumps(payload, separators=(',', ':')),
                    'source': 'fraudshield_rollout',
                    'task_name': task_name,
                })
                step_result = env.step(action)
                observation = step_result.observation
                done = step_result.done
    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)


def public_row_to_training_example(row):
    amount = float(row.get('amount', 0.0) or 0.0)
    transaction_type = str(row.get('transaction_type', row.get('type', 'purchase')))
    location = str(row.get('location', 'unknown'))
    merchant = str(row.get('merchant', row.get('nameDest', 'unknown_merchant')))
    device = str(row.get('device', 'unknown_device'))
    payment_method = str(row.get('payment_method', row.get('transaction_type', 'card')))
    timestamp = str(row.get('timestamp', row.get('step', 'unknown_time')))
    is_fraud = int(row.get('is_fraud', row.get('isFraud', row.get('Class', 0))) or 0)

    high_amount = amount >= 1500
    risky_type = transaction_type.lower() in {'transfer', 'cash_out', 'wire', 'crypto', 'gift_card'}
    risky_location = any(token in location.lower() for token in ['proxy', 'unknown', 'foreign', 'vpn'])
    risky_device = any(token in device.lower() for token in ['emulator', 'root', 'shared', 'new'])

    available = ['merchant_profile', 'customer_profile', 'network_graph', 'payment_trace', 'policy_review']
    visible_observation = {
        'case_id': f"public_case_{abs(hash(str(row.get('transaction_id', merchant)))) % 100000}",
        'task_name': 'medium',
        'current_screen': 'Queue',
        'visible_panels': ['triage_summary'],
        'revealed_evidence': {},
        'linked_case_ids': [],
        'remaining_steps': 6,
        'remaining_sla': 5,
        'note_required': False,
        'allowed_actions': ['review_transaction', 'fetch_customer_profile', 'fetch_merchant_profile', 'fetch_network_graph', 'check_policy', 'resolve_case'],
        'case_summary': {
            'amount_usd': round(amount, 2),
            'queue_reason': f'{transaction_type} transaction flagged for manual review',
            'visible_risk_band': 'review',
            'merchant_region': 'masked',
        },
        'app_context': {
            'item_category': transaction_type,
            'timestamp': timestamp,
            'available_investigations': available,
            'public_signals': {
                'merchant': merchant,
                'device': device,
                'payment_method': payment_method,
                'location': location,
            },
        },
    }

    if is_fraud and (risky_type or high_amount):
        payload = {
            'action_type': 'investigate',
            'investigation_target': 'network_graph' if risky_device or risky_location else 'payment_trace',
            'decision': None,
            'confidence': None,
            'reasoning': 'The visible signals are suspicious, so gather network or payment evidence before routing.',
        }
    elif not is_fraud and amount < 200 and not risky_type:
        payload = {
            'action_type': 'decide',
            'investigation_target': None,
            'decision': 'legitimate',
            'confidence': 0.82,
            'reasoning': 'The visible transaction looks low risk and can be cleared with high confidence.',
        }
    else:
        payload = {
            'action_type': 'investigate',
            'investigation_target': 'merchant_profile' if high_amount else 'customer_profile',
            'decision': None,
            'confidence': None,
            'reasoning': 'The transaction is ambiguous, so inspect merchant or customer history before routing.',
        }

    prompt = (
        'You are a fraud analyst learning how to investigate suspicious payments. '
        'Use visible triage signals to choose the next best FraudShield action. Return JSON only.\n\n'
        f'Visible observation:\n{json.dumps(visible_observation, sort_keys=True)}\n\n'
        f'Valid investigation aliases: {available}.\n'
        'Respond with JSON using this schema: '
        '{"action_type":"investigate|decide","investigation_target":"alias_or_null","decision":"fraud|legitimate|null","confidence":0.0,"reasoning":"one sentence"}'
    )
    return {'text': prompt + '\n' + json.dumps(payload, separators=(',', ':')), 'source': 'public_fraud_data', 'task_name': 'curriculum'}


def build_public_curriculum(max_rows=2500):
    dataset_name = 'Phoenix21/mock_fraud-detection-dataset'
    public_ds = load_dataset(dataset_name, split='train')
    rows = [public_row_to_training_example(row) for row in public_ds.shuffle(seed=42).select(range(min(max_rows, len(public_ds))))]
    print('Loaded public curriculum rows from', dataset_name, 'count=', len(rows))
    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)


public_dataset = build_public_curriculum(max_rows=2500)
fraudshield_dataset = build_fraudshield_rollout_dataset(per_task_episodes=24)
print(public_dataset)
print(fraudshield_dataset)
print(public_dataset[0]['text'][:900])
print(fraudshield_dataset[0]['text'][:900])


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
)
print('Loaded base model and LoRA adapters.')


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

stage1_args = TrainingArguments(
    output_dir='fraudshield-sft-stage1',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
)

stage1_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=public_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=stage1_args,
)

stage1_trainer.train()

stage2_args = TrainingArguments(
    output_dir='fraudshield-sft-stage2',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy='epoch',
    report_to='none',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
)

trainer = SFTTrainer(
    model=stage1_trainer.model,
    tokenizer=tokenizer,
    train_dataset=fraudshield_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=stage2_args,
)

trainer.train()

OUTPUT_DIR = 'trained_policy'
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved trained policy to', OUTPUT_DIR)


In [ ]:
def run_inference(extra_env=None):
    env_vars = os.environ.copy()
    if extra_env:
        env_vars.update(extra_env)
    completed = subprocess.run(
        ['python', 'inference.py'],
        capture_output=True,
        text=True,
        env=env_vars,
        check=True,
    )
    with open('fraudshield_baseline_results.json', 'r', encoding='utf-8') as handle:
        results = json.load(handle)
    return results, completed.stdout

baseline_results, baseline_stdout = run_inference()
trained_results, trained_stdout = run_inference({'LOCAL_MODEL_PATH': OUTPUT_DIR})

comparison_rows = []
for task_name in ('easy', 'medium', 'hard'):
    comparison_rows.append({
        'task': task_name,
        'heuristic_score': baseline_results[task_name]['score'],
        'trained_score': trained_results[task_name]['score'],
        'delta': trained_results[task_name]['score'] - baseline_results[task_name]['score'],
    })

print('Heuristic baseline stdout:
', baseline_stdout)
print('Trained model stdout:
', trained_stdout)
print(json.dumps(comparison_rows, indent=2))


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
loss_points = [(entry['step'], entry['loss']) for entry in log_history if 'step' in entry and 'loss' in entry]

if loss_points:
    xs, ys = zip(*loss_points)
    plt.figure(figsize=(8, 4))
    plt.plot(xs, ys)
    plt.xlabel('training step')
    plt.ylabel('loss')
    plt.title('FraudShield loss curve')
    plt.tight_layout()
    plt.savefig('loss_curve.png')
    plt.close()

reward_like_points = [(idx + 1, row['delta']) for idx, row in enumerate(comparison_rows)]
plt.figure(figsize=(8, 4))
plt.plot([x for x, _ in reward_like_points], [y for _, y in reward_like_points], marker='o')
plt.xticks([1, 2, 3], ['easy', 'medium', 'hard'])
plt.xlabel('task')
plt.ylabel('score delta vs heuristic')
plt.title('FraudShield score improvement by task')
plt.tight_layout()
plt.savefig('reward_curve.png')
plt.close()

summary = {
    'status': 'completed',
    'updated_at': datetime.utcnow().isoformat() + 'Z',
    'trainer': 'Two-stage TRL SFTTrainer with Unsloth LoRA',
    'base_model': MODEL_NAME,
    'public_curriculum_dataset': 'Phoenix21/mock_fraud-detection-dataset',
    'local_model_path': OUTPUT_DIR,
    'baseline': {
        'easy': baseline_results['easy']['score'],
        'medium': baseline_results['medium']['score'],
        'hard': baseline_results['hard']['score'],
        'final_score': baseline_results['final_score'],
    },
    'trained': {
        'easy': trained_results['easy']['score'],
        'medium': trained_results['medium']['score'],
        'hard': trained_results['hard']['score'],
        'final_score': trained_results['final_score'],
    },
    'score_delta': trained_results['final_score'] - baseline_results['final_score'],
    'task_comparison': comparison_rows,
    'artifact_urls': {
        'reward_plot': 'reward_curve.png',
        'loss_plot': 'loss_curve.png',
        'comparison_table': 'training_summary.json',
    },
}

with open('training_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
print('Artifacts saved: reward_curve.png, loss_curve.png, training_summary.json')
